# Lesson 20 — Queue、Event 与 race condition

累计脉冲在离线期间不能丢失，恢复后消费者一次处理一条。先用确定性计数器，再用 `%race_probe` 运行真实 `std::thread` + `std::atomic`。未同步共享变量属于 C++ 未定义行为，不能用一次“看起来正常”证明正确。

**学习顺序：** 先运行 guided，再修改，故意运行 broken，最后自己完成 exercise。答案 Notebook 独立保存。

In [ ]:
from pathlib import Path
project = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "tools/notebook_setup.py").is_file())
exec((project / "tools/notebook_setup.py").read_text())

## 1. Guided

先观察面板输出，再读 C++。

In [ ]:
%%academy_lab 20-sync-and-race guided guided pass
#pragma once
#include "academy/device.hpp"
namespace academy {
    class Controller {
    public:
        explicit Controller(Device& d) : device_(d) {
        }
        void tick() {
            if (device_.input.pulses > observed_)
                pending_ += int(device_.input.pulses - observed_);
            observed_ = device_.input.pulses;
            device_.output.error = device_.input.fault;
            if (device_.input.fault)
                device_.output.state = "fault";
            else if (!device_.input.connected)
                device_.output.state = "offline";
            else if (pending_ > 0) {
                --pending_;
                ++device_.output.events;
                device_.output.state = "processing";
            } else
                device_.output.state = "waiting";
            device_.output.queue_depth = pending_;
        }

    private:
        Device& device_;
        std::uint32_t observed_ = 0;
        int pending_ = 0;
    };
}

## 2. Modify

把每次消费一条改成每次消费两条，预测 queue_depth 如何变化。

In [ ]:
%%academy_lab 20-sync-and-race modify
#pragma once
#include "academy/device.hpp"
namespace academy {
    class Controller {
    public:
        explicit Controller(Device& d) : device_(d) {
        }
        void tick() {
            if (device_.input.pulses > observed_)
                pending_ += int(device_.input.pulses - observed_);
            observed_ = device_.input.pulses;
            device_.output.error = device_.input.fault;
            if (device_.input.fault)
                device_.output.state = "fault";
            else if (!device_.input.connected)
                device_.output.state = "offline";
            else if (pending_ > 0) {
                --pending_;
                ++device_.output.events;
                device_.output.state = "processing";
            } else
                device_.output.state = "waiting";
            device_.output.queue_depth = pending_;
        }

    private:
        Device& device_;
        std::uint32_t observed_ = 0;
        int pending_ = 0;
    };
}

## 3. Broken

下面的检查应该失败；先观察故障证据，再查看分层提示。

In [ ]:
%%academy_lab 20-sync-and-race broken exercise fail
#pragma once
#include "academy/device.hpp"
namespace academy {
    class Controller {
    public:
        explicit Controller(Device& d) : device_(d) {
        }
        void tick() {
            if (device_.input.pulses > observed_)
                pending_ = int(device_.input.pulses - observed_);
            observed_ = device_.input.pulses;
            if (device_.input.connected && pending_ > 0) {
                --pending_;
                ++device_.output.events;
            }
            device_.output.queue_depth = pending_;
        }

    private:
        Device& device_;
        std::uint32_t observed_ = 0;
        int pending_ = 0;
    };
}

## 4. Build

要求：新 pulses 是累计计数；离线时追加 backlog；fault 时不消费；连接后每 tick 只消费一条。

提示 1：新增量是 `current - observed`。提示 2：追加用 `+=`，不是 `=`。提示 3：fault 路径应保留 queue_depth。

In [ ]:
%%academy_lab 20-sync-and-race exercise exercise
#pragma once
#include "academy/device.hpp"
namespace academy {
    class Controller {
    public:
        explicit Controller(Device& d) : device_(d) {
        }
        void tick() { /* TODO: 你的实现 */
        }

    private:
        Device& device_;
    };
}


## 5. 真实 atomic 探针

In [ ]:
%race_probe

## 6. 反思

写下：这个模型在 host 上证明了什么？哪些行为必须买板后重新验证？